In [1]:
import torch

model_path = 'models/dinov2_vits14_reg4_pretrain.pth'
state_dict = torch.load(model_path, map_location='cpu')
print(type(state_dict))

<class 'dict'>


In [2]:
import timm
print(timm.list_models('*dinov2*'))
# Not using timm since there is a slight name mismatch between state dict (reg_token vs register_token)

['vit_base_patch14_dinov2', 'vit_base_patch14_reg4_dinov2', 'vit_giant_patch14_dinov2', 'vit_giant_patch14_reg4_dinov2', 'vit_large_patch14_dinov2', 'vit_large_patch14_reg4_dinov2', 'vit_small_patch14_dinov2', 'vit_small_patch14_reg4_dinov2']


/Users/gaurav/Desktop/Boulder/env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import torch
from functools import partial
import dinov2.models.vision_transformer as vits

model = vits.vit_small(img_size=518, patch_size=14, num_register_tokens=4, init_values=1.0, block_chunks=0)

ckpt = torch.load('models/dinov2_vits14_reg4_pretrain.pth', map_location='cpu')
model.load_state_dict(ckpt)

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
model = model.to(device).eval()
print("Ready on", device)

/Users/gaurav/Desktop/Boulder/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/Users/gaurav/Desktop/Boulder/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/Users/gaurav/Desktop/Boulder/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Ready on mps


In [ ]:
from torchvision import transforms
from PIL import Image
import torch
import matplotlib.pyplot as plt

transform = transforms.Compose([
    transforms.Resize((518, 518)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

img = Image.open('climb holds good.v2i.sam2/train/0ea408b3-a616-4f05-868e-e8b494619a71_jpg.rf.8c6f407b890281b58e6bd270db7c2912.jpg').convert('RGB')
x = transform(img).unsqueeze(0).to(device)  # (1, 3, 518, 518)

attn_output = {}

def hook_fn(module, input, output):
    attn_output['attn'] = output  # shape: (B, num_heads, N, N)

# blocks[-1] is the last transformer block, .attn is the attention module
hook = model.blocks[-1].attn.register_forward_hook(hook_fn) # Calls the function in question every single time a forward() pass happens

with torch.no_grad():
    out = model.forward_features(x)

hook.remove()  # always clean up

attn = attn_output['attn']  # (1, 6, 1374, 1374)
# dim 2/3 = num_tokens = 1 CLS + 4 reg + 1369 patches

In [ ]:
print(attn.shape)
import dinov2.layers as layers #noqa
import inspect # noqa
print(inspect.getsource(layers.MemEffAttention))

torch.Size([1, 1374, 384])
class MemEffAttention(Attention):
    def forward(self, x: Tensor, attn_bias=None) -> Tensor:
        if not XFORMERS_AVAILABLE:
            if attn_bias is not None:
                raise AssertionError("xFormers is required for using nested tensors")
            return super().forward(x)

        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, C // self.num_heads)

        q, k, v = unbind(qkv, 2)

        x = memory_efficient_attention(q, k, v, attn_bias=attn_bias)
        x = x.reshape([B, N, C])

        x = self.proj(x)
        x = self.proj_drop(x)
        return x



In [15]:
attn_layer_module = layers.attention
print(attn_layer_module.XFORMERS_ENABLED)

attn_layer_module.XFORMERS_ENABLED = False

from dinov2.layers import Attention #noqa
print(inspect.getsource(Attention))

False
class Attention(nn.Module):
    def __init__(
        self,
        dim: int,
        num_heads: int = 8,
        qkv_bias: bool = False,
        proj_bias: bool = True,
        attn_drop: float = 0.0,
        proj_drop: float = 0.0,
    ) -> None:
        super().__init__()
        self.dim = dim
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = head_dim**-0.5

        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = attn_drop
        self.proj = nn.Linear(dim, dim, bias=proj_bias)
        self.proj_drop = nn.Dropout(proj_drop)

    def init_weights(
        self, init_attn_std: float | None = None, init_proj_std: float | None = None, factor: float = 1.0
    ) -> None:
        init_attn_std = init_attn_std or (self.dim**-0.5)
        init_proj_std = init_proj_std or init_attn_std * factor
        nn.init.normal_(self.qkv.weight, std=init_attn_std)
        nn.init.normal_(self.proj.weight, std=init_proj_std)
    

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn.functional as F

# --- get CLS->patch attention ---
cls_attn = attn[0, :, 0, 5:]          # (6, 1369) attn[0, :, 0, 5:] — row 0 is CLS, and you skip the first 5 columns (1 CLS + 4 registers) to get only the patch-to-patch attention.
cls_attn_mean = cls_attn.mean(0)       # (1369,)

# --- reshape to spatial grid and upsample to image size ---
grid_size = 518 // 14  # 37
attn_map = cls_attn_mean.reshape(1, 1, grid_size, grid_size)
attn_map = F.interpolate(attn_map, size=(518, 518), mode='bilinear', align_corners=False)
attn_map = attn_map.squeeze().cpu().numpy()  # (518, 518)

# --- normalize to [0, 1] for colormap ---
attn_map = (attn_map - attn_map.min()) / (attn_map.max() - attn_map.min())

# --- overlay ---
original = img.resize((518, 518))  # PIL image, already loaded earlier

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(original)
axes[0].set_title('Original')
axes[0].axis('off')

axes[1].imshow(attn_map, cmap='inferno')
axes[1].set_title('Attention Map')
axes[1].axis('off')

axes[2].imshow(original)
axes[2].imshow(attn_map, cmap='inferno', alpha=0.5)
axes[2].set_title('Overlay')
axes[2].axis('off')

plt.tight_layout()
plt.savefig('attention_overlay.png', dpi=150, bbox_inches='tight')
plt.show()

IndexError: too many indices for tensor of dimension 3

In [ ]:
'''
Register tokens are a DINOv2-specific addition from a 2023 paper. The problem they solve: in vanilla ViTs, certain patch tokens in "uninformative" regions (smooth backgrounds, uniform areas) get hijacked by the model to store global information — you can see it as artifact patches with weirdly high attention that don't correspond to anything visual. This pollutes both the patch features and the attention maps.
Register tokens are 4 extra learnable tokens (also 384-dim) inserted between CLS and the patches. They act as "scratchpad" — the model offloads that global bookkeeping onto them instead of corrupting patch tokens. They have no spatial meaning, you discard them for visualisation.
'''

# Try visualizing this to see what exactly happens!